# Q43: What are the main difficulties when training GANs?
**Topic:** Computer-vision | **Difficulty:** Hard | **Time:** 12 min
**Sheet:** Grind75ML

---

## Question
What are the main difficulties when training GANs?

## Detailed Answer

**GANs** (Generative Adversarial Networks) consist of a Generator (G) and a Discriminator (D) trained in a minimax game:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

This adversarial setup creates several training challenges:

### 1. Mode Collapse
**Problem:** Generator produces only a few types of outputs, ignoring the diversity of the real data.

**Example:** A face GAN only generating female faces with blonde hair.

**Solutions:**
- **Minibatch discrimination**: Let D see batches and penalize low diversity
- **Unrolled GANs**: G optimizes against future D updates
- **WGAN/WGAN-GP**: Better loss landscape avoids collapse
- **StyleGAN's diversity regularization**: Path length regularization

### 2. Training Instability (Oscillation)
**Problem:** G and D don't converge — loss oscillates, training diverges.

**Why:** The minimax game may not have a stable Nash equilibrium in practice.

**Solutions:**
- **Two-timescale update rule (TTUR)**: Different learning rates for G and D
- **Spectral normalization**: Constrain D's Lipschitz constant
- **Progressive growing**: Start with low-res, gradually add layers (ProGAN)
- **Learning rate scheduling**: Cosine annealing, warmup

### 3. Vanishing Gradients
**Problem:** When D becomes too strong, the gradient signal for G vanishes.

$$\text{If } D(G(z)) \approx 0: \nabla_G \log(1 - D(G(z))) \approx 0$$

**Solutions:**
- Use **non-saturating loss**: $\max_G \log D(G(z))$ instead of $\min_G \log(1-D(G(z)))$
- **WGAN**: Uses Wasserstein distance (provides gradient even when distributions don't overlap)
- **Hinge loss**: Better gradient properties

### 4. Evaluation Difficulty
**Problem:** No single metric captures both quality and diversity of generated samples.

**Metrics:**
- **FID (Fréchet Inception Distance)**: Compares statistics of real vs generated feature distributions (lower is better)
- **IS (Inception Score)**: Measures quality and diversity (higher is better), but less reliable
- **KID (Kernel Inception Distance)**: Unbiased alternative to FID
- **LPIPS**: Perceptual similarity metric

### 5. Hyperparameter Sensitivity
**Problem:** GANs are extremely sensitive to learning rate, architecture, batch size, etc.

**Best practices:**
- Use Adam with β1=0.0, β2=0.99 (WGAN-GP defaults)
- Train D more than G (typically 1:1 or 5:1 D:G ratio)
- Use batch size 32-64
- Apply spectral normalization in D

### 6. Checkerboard Artifacts
**Problem:** Transpose convolutions create checkerboard patterns in generated images.

**Solutions:**
- Use **nearest-neighbor upsampling + conv** instead of transposed conv
- **Sub-pixel convolution** (PixelShuffle)

### Summary Table:
| Issue | Impact | Best Fix |
|-------|--------|----------|
| Mode collapse | Low diversity | WGAN-GP, minibatch discrimination |
| Training instability | Divergence | Spectral norm, TTUR |
| Vanishing gradients | G stops learning | WGAN, non-saturating loss |
| Evaluation | Hard to compare models | FID (standard metric) |
| Hyperparameter sensitivity | Unpredictable behavior | Established recipes (BigGAN, StyleGAN) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def vanilla_gan_loss(D_real: np.ndarray, D_fake: np.ndarray) -> tuple[float, float]:
    """Original GAN loss (Jensen-Shannon Divergence based)."""
    eps = 1e-12
    d_loss = -np.mean(np.log(D_real + eps) + np.log(1 - D_fake + eps))
    g_loss = -np.mean(np.log(D_fake + eps))  # non-saturating version
    return d_loss, g_loss


def wgan_loss(D_real: np.ndarray, D_fake: np.ndarray) -> tuple[float, float]:
    """Wasserstein GAN loss (Earth Mover Distance based)."""
    d_loss = -(np.mean(D_real) - np.mean(D_fake))  # maximize E[D(x)] - E[D(G(z))]
    g_loss = -np.mean(D_fake)  # maximize E[D(G(z))]
    return d_loss, g_loss


def hinge_loss(D_real: np.ndarray, D_fake: np.ndarray) -> tuple[float, float]:
    """Hinge loss (used in BigGAN, SAGAN)."""
    d_loss = np.mean(np.maximum(0, 1 - D_real)) + np.mean(np.maximum(0, 1 + D_fake))
    g_loss = -np.mean(D_fake)
    return d_loss, g_loss


# --- Compare loss landscapes ---
D_fake_range = np.linspace(0.01, 0.99, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, loss_fn, name in [
    (axes[0], vanilla_gan_loss, 'Vanilla GAN'),
    (axes[1], wgan_loss, 'WGAN'),
    (axes[2], hinge_loss, 'Hinge Loss')
]:
    g_losses = [loss_fn(np.array([0.9]), np.array([d]))[1] for d in D_fake_range]
    ax.plot(D_fake_range, g_losses, 'b-', linewidth=2)
    ax.set_xlabel('D(G(z))')
    ax.set_ylabel('Generator Loss')
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

plt.suptitle('Generator Loss vs Discriminator Output', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key: Vanilla GAN has vanishing gradient when D(G(z))→0")
print("WGAN and Hinge loss maintain useful gradients throughout")

## Key Takeaways
- **Mode collapse** and **training instability** are the two biggest challenges
- **WGAN-GP** (Wasserstein + Gradient Penalty) is the most stable GAN variant
- Always monitor **FID** during training — loss curves are unreliable in GANs
- Modern approaches (diffusion models) have largely replaced GANs for image generation due to these instability issues

In [2]:
st = set()

In [ ]:
arr=((1,3),(3,2))
arr = (arr)+ (2, 3)
st.add(arr)
st

{((1, 3), (3, 2)), ((1, 3), (3, 2), (2, 3))}